# Caso B — Combos · Modelado y evaluación

> **Objetivo.** Segmentar tiendas y minar reglas de co-compra para proponer los mejores combos por cluster, con su lift esperado.

> **Entradas.** Tabla maestra de líneas de ticket + matriz de cestas, vía catálogo.

> **Salidas.** Perfiles y clusters de tienda, selección de k, reglas de asociación, combos propuestos y grafo de co-compra.

> **Cómo ejecutar.** `Restart & Run All`; determinista. Requiere el extra `caso_b` (`uv sync --extra caso_b`). Corre en paralelo al pipeline `caso_b`.

## 1. Datos: líneas de ticket y cestas

Cruce detalle ⨝ tickets ⨝ catálogo; de ahí la matriz booleana ticket×producto.

In [ ]:
from pathlib import Path
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project
from tostao_ml.cases import masters

PROJECT = Path.cwd().parents[1] if Path.cwd().name.startswith('caso') else Path.cwd()
bootstrap_project(PROJECT)
with KedroSession.create(project_path=PROJECT) as session:
    catalog = session.load_context().catalog
    master_b, _ = masters.build_master_b(
        catalog.load('b_tickets'), catalog.load('b_detalle_tickets'),
        catalog.load('b_catalogo_productos'))
baskets = masters.build_baskets_b(master_b)
print('Líneas:', master_b.shape, '| Cestas:', baskets.shape)
master_b.head()

## 2. Estrategia (aprendizaje no supervisado)

No hay variable objetivo: se busca **estructura** (segmentos de tienda) y **patrones** de co-compra.

- **Clustering:** K-Means sobre el perfil de compra; se compara con Aglomerativo y se elige **k por máxima silhouette** (no a dedo).
- **Reglas:** FP-Growth por cluster, filtradas por `lift > 1` y soporte mínimo.
- **Grafo (complementario):** centralidad de productos como red de co-ocurrencia.

In [ ]:
from tostao_ml.cases.caso_b import run_case_b

result = run_case_b(master_b, baskets, tune=True)

## 3. Segmentación de tiendas

Perfil por tienda + cluster asignado; el barrido de k y la comparación de algoritmos justifican la elección.

In [ ]:
result.store_profiles.join(result.store_clusters).round(3).reset_index()

In [ ]:
if result.k_selection is not None:
    display(result.k_selection)
if result.clustering_comparison is not None:
    display(result.clustering_comparison)

In [ ]:
from tostao_ml.framework.viz import eda as eda_viz

eda_viz.pareto(result.store_clusters.value_counts().sort_index(), name='cluster').show()

## 4. Reglas de asociación (co-compra)

Ordenadas por lift: cuántas veces más de lo esperado se compran juntos dos productos.

In [ ]:
rules = result.rules.assign(
    antecedente=result.rules['antecedents'].map(lambda s: ', '.join(sorted(s))),
    consecuente=result.rules['consequents'].map(lambda s: ', '.join(sorted(s))))
rules[['antecedente', 'consecuente', 'support', 'confidence', 'lift']].head(20).round(4)

## 5. Combos propuestos

Top combos por cluster con precio con descuento y lift esperado.

In [ ]:
top = result.combos.sort_values('lift', ascending=False).head(12)
combo_lift = top.assign(combo=top['producto_a'] + ' + ' + top['producto_b']).set_index('combo')['lift']
eda_viz.pareto(combo_lift, name='combo (lift)').show()
result.combos.head(15)

## 6. Grafo de co-compra (enfoque complementario)

Productos «hub» por centralidad: conectan la red de co-compra.

In [ ]:
if result.graph_centrality is not None:
    display(result.graph_centrality)

## 7. Lectura interpretada

Validez de los clusters y robustez de las reglas, en palabras.

In [ ]:
from IPython.display import Markdown
from tostao_ml.cases import storytelling as st

Markdown(st.interpret_model_b(result).to_markdown())

## Conclusión

La segmentación es moderada pero clara (silhouette ~0.32) y las reglas tienen lift alto con soporte suficiente: los combos propuestos reflejan co-compra genuina, no coincidencia, y priorizan qué activar por segmento.